# QLoRA training (Colab)

Run this on a Colab GPU runtime (Runtime > Change runtime type > GPU).

This notebook runs the **real 4-bit QLoRA** training path. It reuses the exact same
code as the repo (`train/qlora_train.py`) so the Colab run and the local CPU smoke
test can never drift: when CUDA is present, `qlora_train.py` loads the model in 4-bit
with Unsloth and trains with TRL's `SFTTrainer`; on CPU it falls back to plain `peft`
LoRA (that fallback is what proved the plumbing on Day 2).

Prereqs (Day 2, done): Behavior Spec written, data pipeline in `data/`, eval harness
in `eval/`, and the end-to-end smoke test passing. Day 3 uses this for the first real
training run on the v1 dataset.

## 1. Install environment

In [1]:
!pip install -q unsloth trl datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

## 2. Load base model and confirm GPU inference works

In [2]:
from unsloth import FastLanguageModel

MODEL_ID = "Qwen/Qwen3-4B"  # tune target upgraded 0.6B -> 4B to clear the accuracy floor
                            # (0.6B was capacity-bound: litmus accuracy 5/12). 4B is the CAP.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    load_in_4bit=True,   # QLoRA 4-bit: ~2.2GB weights; full run peaks ~6-8GB, fits a free T4 (16GB)
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-1): 2 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm

In [3]:
messages = [{"role": "user", "content": "In one sentence, what is a QLoRA fine-tune?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

<think>
Okay, the user is asking for a one-sentence definition of QLoRA fine-tuning. Let me start by recalling what QLoRA is. I know that QLoRA stands for Quantum LoRA, but wait, no, that's not right. Wait, QLoRA is actually a technique for training large language models with lower computational resources. It's a combination of quantization, low-rank adaptation, and a more efficient training method.

So, the key components here


## 3. Train on the generated dataset (real QLoRA)

Clone the repo so the notebook runs the *same* code and dataset as everything else,
then invoke the training module. On a GPU runtime this takes the Unsloth 4-bit QLoRA
path automatically.

In [ ]:
# Get the repo so the notebook runs the *same* code + dataset as everything else.
%cd /content
![ -d SmallLearningModel ] || git clone https://github.com/Elitelord/SmallLearningModel.git
%cd SmallLearningModel
!git pull  # make sure you have the latest committed dataset + code

# Real QLoRA run on the v4 gold dataset (Qwen3-4B). Uses Unsloth 4-bit + TRL SFTTrainer on GPU.
# --data is the combined v4 training file (227 synthetic + 1 real); must be COMMITTED & PUSHED
# (Colab clones from GitHub, so uncommitted local files are NOT visible here).
# (For a plumbing smoke test, use --data data/generated_v0.jsonl and add --max-steps 10.)
!python -m train.qlora_train \
    --data data/v4/gold_v4_final.jsonl \
    --adapter-name v4 \
    --epochs 3 \
    --lora-r 16 --lora-alpha 32 \
    --batch-size 8 --grad-accum 2 --lr 2e-4

## 4. Base-vs-tuned eval

Score the base model and the freshly trained adapter on the held-out concepts with the
identical minimal prompt, using the reused litmus FK scorer + accuracy judge. Set your
OpenAI key first (`import os; os.environ["OPENAI_API_KEY"]="..."`), or pass `--no-judge`
for an FK-only run.

In [ ]:
!python -m eval.base_vs_tuned --adapter train/adapters/v1 --limit 12